# 📐 Cuaderno 01 — Interpolación

**Métodos Numéricos · Unidad 4: Ajuste de curvas e interpolación**

Tema 4.1. Cada método sigue el mismo camino: **explicación → a mano → programar → laboratorio → preguntas**.

In [ ]:
#@title ⚙️ Entorno — ejecuta esta celda PRIMERO
# ---------------------------------------------------------------
# En tu computadora usa los archivos locales, sin repositorio.
# En Google Colab pide UNA vez el enlace de tu fork, lo guarda en tu
# Drive y a partir de ahí lo reutiliza solo.
# ---------------------------------------------------------------
import json
import os
import subprocess
import sys

CUADERNO = 'u4_01_interpolacion'
TAREA_ID = 'U4_01_Interpolacion'
REPO_NOMBRE = "U4_Repo"
GUARDADO = os.path.join("CursoMN", "u4_repo.json")   # relativo a MyDrive

try:
    import google.colab  # noqa: F401
    ES_COLAB = True
except ImportError:
    ES_COLAB = False


def _raiz_local():
    """Sube desde la carpeta actual hasta encontrar lib/robomat.py."""
    candidatos = []
    nb = globals().get("__vsc_ipynb_file__")
    if nb:
        candidatos.append(os.path.dirname(os.path.abspath(nb)))
    candidatos.append(os.getcwd())
    for base in candidatos:
        b = base
        for _ in range(6):
            if os.path.exists(os.path.join(b, "lib", "robomat.py")):
                return b
            padre = os.path.dirname(b)
            if padre == b:
                break
            b = padre
    return None


def _normaliza(enlace):
    enlace = (enlace or "").strip().rstrip("/")
    if enlace.endswith(".git"):
        enlace = enlace[:-4]
    if not enlace.startswith("https://github.com/"):
        raise ValueError("Necesito un enlace https://github.com/USUARIO/"
                         + REPO_NOMBRE + " (tu fork).")
    return enlace

# Repositorio del curso (upstream) y carpetas que el alumno nunca edita:
# se refrescan en cada ejecución para que el material y el motor nunca estén
# atrasados, aunque el fork se haya quedado viejo.
CURSO = "https://github.com/ITH-MGL-MN/" + REPO_NOMBRE
MATERIAL = ("lib", "grader", "config", "docs", "ejercicios", "tools")


def _git(*args):
    """Corre git dentro del clon. Devuelve (funcionó, salida)."""
    r = subprocess.run(["git", "-C", REPO] + list(args),
                       capture_output=True, text=True)
    return r.returncode == 0, (r.stdout + r.stderr).strip()


def actualizar():
    """Trae tus commits de GitHub y refresca el material del curso.

    El clon de Drive es desechable: tus respuestas viven en el cuaderno.
    Primero se actualiza el fork (lo que editaste en GitHub) y después se
    copia el material del curso desde `upstream/main`. Si tienes cambios
    locales en el clon, no se tocan: se avisa y se sigue.
    """
    ok, _ = _git("rev-parse", "--is-inside-work-tree")
    if not ok:
        print("⚠️ La carpeta no es un repositorio git: uso los archivos tal cual.")
        return
    _git("remote", "add", "upstream", CURSO + ".git")
    ok, salida = _git("fetch", "--all", "--prune", "--quiet")
    if not ok:
        print("⚠️ No pude conectar con GitHub; sigo con lo que hay en Drive.")
        return

    _git("checkout", "--", *MATERIAL)          # limpia el refresco anterior
    _, rama = _git("rev-parse", "--abbrev-ref", "HEAD")
    rama = rama or "main"
    ok, salida = _git("merge", "--ff-only", "origin/" + rama)
    if ok:
        print("Fork        : al día con tus commits de GitHub")
    else:
        print("⚠️ Tu fork no avanza en línea recta (tienes commits propios")
        print("   o cambios locales): lo dejo como está y solo refresco el")
        print("   material del curso.")

    ok, salida = _git("checkout", "upstream/main", "--", *MATERIAL)
    if ok:
        _git("reset", "--quiet")
        _, sha = _git("rev-parse", "--short", "upstream/main")
        print("Material    : actualizado al curso", sha)
    else:
        print("⚠️ No pude refrescar el material del curso:", salida.splitlines()[:1])


if not ES_COLAB:
    REPO = _raiz_local()
    if REPO is None:
        raise RuntimeError(
            "No encontré la raíz del repositorio (lib/robomat.py).\n"
            "Abre el cuaderno dentro de U4_Repo o ejecútalo desde esa carpeta.")
    print("Modo        : local (sin repositorio)")
    print("Repositorio :", REPO)
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive"
    ruta_json = os.path.join(DRIVE, GUARDADO)

    guardado = {}
    if os.path.exists(ruta_json):
        try:
            with open(ruta_json, encoding="utf-8") as f:
                guardado = json.load(f)
        except (OSError, json.JSONDecodeError) as exc:
            raise RuntimeError(
                f"No se pudo leer la configuracion de Drive: {ruta_json}"
            ) from exc

    enlace = (guardado.get("repo") or "").strip()
    if enlace:
        print("✅ Repositorio guardado:", enlace)
        print("   (si cambió, edita la caja de abajo y pulsa el botón)")
    else:
        try:
            enlace = input("Pega el enlace de TU fork de U4_Repo: ")
        except Exception:
            enlace = ""
        enlace = (enlace or "").strip()
        if not enlace:
            raise RuntimeError(
                "Sin repositorio no puedo continuar en Colab.\n"
                "Haz fork del repositorio del curso y pega su enlace.")

    enlace = _normaliza(enlace)
    REPO = os.path.join(DRIVE, "CursoMN", REPO_NOMBRE)

    if not os.path.exists(REPO):
        print("Clonando tu repositorio…")
        subprocess.run(["git", "clone", enlace + ".git", REPO], check=True)
    actualizar()

    with open(ruta_json, "w", encoding="utf-8") as f:
        json.dump({"repo": enlace}, f, indent=1)
    print("Repositorio :", REPO)
    print("Actualizar  : vuelve a ejecutar esta celda cuando subas cambios a GitHub")

# Instala dependencias del repositorio solo si falta alguna.
import importlib.util

_modulos_requeridos = ("numpy", "scipy", "sympy", "matplotlib", "yaml", "requests")
_faltantes = [nombre for nombre in _modulos_requeridos
              if importlib.util.find_spec(nombre) is None]
if _faltantes:
    requisitos = os.path.join(REPO, "requirements.txt")
    if not os.path.isfile(requisitos):
        raise FileNotFoundError("Falta requirements.txt en el repositorio.")
    print("Instalando dependencias faltantes:", ", ".join(_faltantes))
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", requisitos],
                   check=True)

# Indica al motor empaquetado dónde está la configuración del repositorio.
os.environ["MN_REPO"] = REPO

# --- Motor de calificación -------------------------------------------
# Se usa el código fuente si está (rama `test`, para depurar); si no, el blob
# empaquetado de grader/, que es lo que recibe el alumno.
_ruta_modulos = os.path.join(REPO, "lib", "mn_motor")
_ruta_blob = os.path.join(REPO, "grader", CUADERNO + ".txt")

_lib = os.path.join(REPO, "lib")
if _lib not in sys.path:
    sys.path.insert(0, _lib)          # ajuste_curvas y, si está, mn_motor

if os.path.isfile(os.path.join(_ruta_modulos, "__init__.py")):
    import mn_motor as _motor
    globals().update({n: getattr(_motor, n) for n in _motor.__all__})
    print("Motor       : módulos de lib/mn_motor (modo profesor)")
else:
    import base64
    import zlib

    if not os.path.exists(_ruta_blob):
        raise FileNotFoundError(
            "No encuentro el motor: falta grader/" + CUADERNO + ".txt\n"
            "Clona el repositorio completo (incluye la carpeta grader/).")
    with open(_ruta_blob, encoding="utf-8") as f:
        codigo = zlib.decompress(base64.b64decode(f.read().strip())).decode("utf-8")
    _MOTOR = {"__name__": "mn_motor_blob", "__file__": "<blob>"}
    exec(compile(codigo, "<blob>", "exec"), _MOTOR)
    globals().update({n: _MOTOR[n] for n in _MOTOR["__all__"]})
    print("Motor       : blob verificado de grader/")

usar_cuaderno(CUADERNO, TAREA_ID)

# --- Preludio estilo MATLAB y herramientas de gráficas ---------------
with open(os.path.join(REPO, "lib", "robomat.py"), encoding="utf-8") as f:
    exec(f.read(), globals())

if REPO not in sys.path:
    sys.path.insert(0, REPO)          # para importar ejercicios/
sys.path.append(os.path.join(REPO, "tools"))
import importlib
import visualizar
importlib.reload(visualizar)

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["figure.dpi"] = 110
print("Preludio    : lib/robomat.py · tools/visualizar.py")
print("\nListo. Ejecuta la celda de identificación.")

In [ ]:
#@title 👤 Identificación (número de control)
# ---------------------------------------------------------------
# En Colab se toma tu correo institucional; en local se escribe uno
# de prueba. De ahí sale el número de control que siembra tus datos.
# ---------------------------------------------------------------
if ES_COLAB:
    import requests

    from google.colab import auth
    import google.auth
    import google.auth.transport.requests

    auth.authenticate_user()
    creds, _ = google.auth.default()
    creds.refresh(google.auth.transport.requests.Request())
    info = requests.get(
        "https://www.googleapis.com/drive/v3/about?fields=user",
        headers={"Authorization": "Bearer " + creds.token}, timeout=20).json()
    cuenta = info.get("user", {})
    email = cuenta.get("emailAddress", "")
    nombre = cuenta.get("displayName", "")
else:
    email = "m16330887@hermosillo.tecnm.mx"   # <-- cambia por el tuyo
    nombre = ""

if not (email and "@" in email and email.lower().endswith(".tecnm.mx")):
    from IPython.display import HTML, display
    display(HTML(
        '<p style="color:#b00020;background:#ffe0e0;padding:12px;'
        'border-radius:8px;font-weight:bold">⛔ Usa tu correo @….tecnm.mx</p>'))
    raise RuntimeError("Necesito tu correo institucional para sembrar tus datos.")

print("Alumno :", email)
print("NC     :", extraer_nc(email))
MI_TAREA = generar_tarea(email, nombre)
print("Cuaderno:", MI_TAREA["cuaderno"],
      "·", len(MI_TAREA["preguntas"]), "preguntas")

---

## Antes de interpolar: polinomios de Taylor

Una función suave puede aproximarse cerca de un punto $a$ mediante una suma
infinita de potencias construida con sus derivadas en ese punto. La **serie de
Taylor** es

$$f(x)=\sum_{k=0}^{\infty}\frac{f^{(k)}(a)}{k!}(x-a)^k,$$

cuando la serie converge a $f(x)$ en infinitas iteraciones.

En la práctica se conservan solo los primeros $n+1$ términos: el **polinomio de
Taylor** de grado $n$,

$$T_n(x)=\sum_{k=0}^{n}\frac{f^{(k)}(a)}{k!}(x-a)^k.$$

Por ejemplo, para $e^x$ alrededor de $a=0$, $T_2(x)=1+x+\frac{x^2}{2}$. En
$x=0.2$ da $1.22$, mientras que $e^{0.2}\approx1.2214$; la diferencia es el
error de truncamiento algorítmico. Bajo las condiciones del teorema de Taylor,
ese error puede escribirse como

$$R_n(x)=f(x)-T_n(x)=\frac{f^{(n+1)}(\xi)}{(n+1)!}(x-a)^{n+1},$$

para algún punto $\xi$ entre $a$ y $x$.

**Conexión con esta unidad:** Taylor usa derivadas en un punto para aproximar
localmente; la **interpolación** usa valores conocidos en varios nodos y
construye un polinomio que pasa por ellos. Aquí trabajaremos con esta segunda
idea.

---

## Método 1 · Interpolación lineal

Es unir dos puntos con una recta. Por triángulos semejantes,

$$f_1(x)=f(x_0)+\frac{f(x_1)-f(x_0)}{x_1-x_0}(x-x_0) \tag{18.2}$$

El cociente es la pendiente y, además, una aproximación a la primera derivada.
Cuanto más cerca estén los nodos, mejor representa la recta a la curva.

**Dos salidas.** La función devuelve `(y, error)`. El error sólo se puede
estimar si hay un tercer dato:

$$R_1\cong f[x_2,x_1,x_0]\,(x-x_0)(x-x_1) \tag{18.18}$$

Mientras no escribas `error_estimado_interpolacion` (método 3), regresa
`(y, None)`.

![Figura 18.2 del libro: interpolación lineal](https://raw.githubusercontent.com/ITH-MGL-MN/U4_Repo/refs/heads/main/docs/figuras/libro/fig18-2_interpolacion_lineal.jpg)

In [ ]:
#@title ✏️ Actividad a mano · Interpolación lineal
EJ_1 = mano_enunciado("INTERP_LINEAL")

**🔢 Resolución paso a paso.**

1. Identifica el intervalo $[x_0,x_1]$ que contiene a $x$.
2. Anota $y_0 = f(x_0)$ y $y_1 = f(x_1)$.
3. Calcula la pendiente $m=\dfrac{y_1-y_0}{x_1-x_0}$.
4. Evalúa $y=y_0+m(x-x_0)$.

In [ ]:
#@title ✅ Comprueba tus valores a mano · Interpolación lineal
# Haz la cuenta a mano y escribe abajo tu resultado con 4 decimales:

y = 0.0
mano_comprueba("INTERP_LINEAL", y)

### 💻 Programa la función

Abre `ejercicios/interpolacion/interpolacion_lineal.py` en GitHub o copia la plantilla aquí abajo.

In [ ]:
#@title 💻 Tu función · Interpolación lineal
from ejercicios.interpolacion.interpolacion_lineal import *

In [ ]:
#@title ▶️ Prueba tu función con un ejemplo
practica("INTERP_LINEAL")

In [ ]:
#@title 📊 Laboratorio · Interpolación lineal
visualizar.laboratorio("INTERP_LINEAL", EJ_1)

In [ ]:
#@title 🧪 Pregunta 1 · Interpolación lineal
# Cuando tu función ya funcione, contéstala aquí.
pregunta(1)

In [ ]:
#@title 🧪 Pregunta 2 · Interpolación lineal
# Cuando tu función ya funcione, contéstala aquí.
pregunta(2)

---

## Método 2 · Interpolación cuadrática

Con tres puntos se ajusta el único polinomio de segundo grado:

$$f_2(x)=b_0+b_1(x-x_0)+b_2(x-x_0)(x-x_1) \tag{18.3}$$

Los coeficientes salen de dos pendientes:

$$m_1=\frac{y_1-y_0}{x_1-x_0} \qquad\text{y}\qquad
m_2=\frac{y_2-y_1}{x_2-x_1}$$

$$b_0=y_0 \tag{18.4}$$
$$b_1=m_1 \tag{18.5}$$
$$b_2=\frac{m_2-m_1}{x_2-x_0} \tag{18.6}$$

Los dos primeros términos son la recta; $b_2$ mide cómo cambia la pendiente y
aporta la curvatura. Todo se calcula dentro de la misma función: no hace falta
un paso aparte para los coeficientes.

**Dos salidas.** Devuelve `(y, error)`, y el error sólo aparece con un cuarto
dato:

$$R_2\cong f[x_3,x_2,x_1,x_0]\,(x-x_0)(x-x_1)(x-x_2) \tag{18.18}$$

![Figura 18.4 del libro: parábola por tres puntos](https://raw.githubusercontent.com/ITH-MGL-MN/U4_Repo/refs/heads/main/docs/figuras/libro/fig18-4_ln2_cuadratica.jpg)

In [ ]:
#@title ✏️ Actividad a mano · Interpolación cuadrática
EJ_2 = mano_enunciado("INTERP_CUADRATICA")

**🔢 Resolución paso a paso.**

1. Anota los tres nodos $(x_0,y_0)$, $(x_1,y_1)$, $(x_2,y_2)$ y el punto $x$.
2. Calcula las dos pendientes entre tramos consecutivos:
   $m_1=\dfrac{y_1-y_0}{x_1-x_0}$ y $m_2=\dfrac{y_2-y_1}{x_2-x_1}$.
3. Coeficientes: $b_0=y_0$, $b_1=m_1$ y $b_2=\dfrac{m_2-m_1}{x_2-x_0}$.
4. Evalúa $y=b_0+b_1(x-x_0)+b_2(x-x_0)(x-x_1)$.

In [ ]:
#@title ✅ Comprueba tus valores a mano · Interpolación cuadrática
# Haz la cuenta a mano y escribe abajo tu resultado con 4 decimales:

y = 0.0
mano_comprueba("INTERP_CUADRATICA", y)

### 💻 Programa la función

Abre `ejercicios/interpolacion/interpolacion_cuadratica.py` en GitHub o copia la plantilla aquí abajo.
El método completo debe **llamar** a sus piezas: no repitas la cuenta.

In [ ]:
#@title 💻 Tu función · Interpolación cuadrática
from ejercicios.interpolacion.interpolacion_cuadratica import *

In [ ]:
#@title ▶️ Prueba tu función con un ejemplo
practica("INTERP_CUADRATICA")

In [ ]:
#@title 📊 Laboratorio · Interpolación cuadrática
visualizar.laboratorio("INTERP_CUADRATICA", EJ_2)

In [ ]:
#@title 🧪 Pregunta 3 · Interpolación cuadrática
# Cuando tu función ya funcione, contéstala aquí.
pregunta(3)

In [ ]:
#@title 🧪 Pregunta 4 · Interpolación cuadrática
# Cuando tu función ya funcione, contéstala aquí.
pregunta(4)

---

## Método 3 · Error con un punto extra

Un dato más de los necesarios sirve para **estimar** el error sin
conocer la función:

$$R_n\cong f[x_{n+1},x_n,\dots,x_0]\prod_{i=0}^{n}(x-x_i) \tag{18.18}$$

Es la diferencia entre la predicción de grado $n+1$ y la de grado $n$
(ecuación 18.19). Cuando termines esta función, vuelve a tus dos métodos y
activa el cálculo del error.

![Figura 18.6 del libro: con un nodo extra se mide el error](https://raw.githubusercontent.com/ITH-MGL-MN/U4_Repo/refs/heads/main/docs/figuras/libro/fig18-6_ln2_cubica.jpg)

In [ ]:
#@title ✏️ Actividad a mano · Error con un punto extra
EJ_3 = mano_enunciado("ERROR_INTERP")

**🔢 Resolución paso a paso.**

1. Ordena los cuatro nodos de menor a mayor en $x$.
2. Calcula las pendientes de primer orden $f[x_i,x_j]=\dfrac{y_j-y_i}{x_j-x_i}$.
3. Con ellas arma las de segundo orden y, al final, la de tercer orden
   $f[x_3,x_2,x_1,x_0]$.
4. Multiplica esa diferencia por $(x-x_0)(x-x_1)(x-x_2)$: ése es $R_2$.

In [ ]:
#@title ✅ Comprueba tus valores a mano · Error con un punto extra
# Haz la cuenta a mano y escribe abajo tu resultado con 4 decimales:
R = 0.0

mano_comprueba("ERROR_INTERP", R)

### 💻 Programa la función

Abre `ejercicios/interpolacion/error_estimado_interpolacion.py` en GitHub o copia la plantilla aquí abajo.
El método completo debe **llamar** a sus piezas: no repitas la cuenta.

In [ ]:
#@title 💻 Tu función · Error con un punto extra
from ejercicios.interpolacion.error_estimado_interpolacion import *

In [ ]:
#@title ▶️ Prueba tu función con un ejemplo
practica("ERROR_INTERP")

In [ ]:
#@title 📊 Laboratorio · Error con un punto extra
visualizar.laboratorio("ERROR_INTERP", EJ_3)

In [ ]:
#@title 🧪 Pregunta 5 · Error con un punto extra
# Cuando tu función ya funcione, contéstala aquí.
pregunta(5)

In [ ]:
#@title 🧪 Pregunta 6 · Error con un punto extra
# Cuando tu función ya funcione, contéstala aquí.
pregunta(6)

In [ ]:
#@title 🔗 Conecta el error con tus métodos
# Ya tienes error_estimado_interpolacion. Vuelve a tus dos funciones y activa
# el cálculo del error: con el punto extra deben devolver (y, error).
d = EJ_3["datos"]
x0, x1, x2 = d["nodos"]
x3 = d["x_todos"][3]
y0, y1, y2, y3 = d["y"]

print("nodos:", d["x_todos"], "· valores:", d["y"])
print("punto a estimar: x =", d["x"], "\n")

try:
    print("cuadrática sin cuarto dato:", interpolacion_cuadratica(x0, y0, x1, y1, x2, y2, d["x"]))
    print("cuadrática con cuarto dato:", interpolacion_cuadratica(
        x0, y0, x1, y1, x2, y2, d["x"], x3=x3, y3=y3))
except NotImplementedError as exc:
    print("⏳ Todavía falta una de tus funciones:", exc)
except TypeError as exc:
    print("⚠️ Revisa la firma de tu función (debe aceptar x3 e y3):", exc)

print("\nEl valor de referencia de tu actividad a mano no se muestra aquí:")
print("compáralo con lo que reportaste en la celda ✅ del método 3.")

In [ ]:
#@title ✅ 1 · Verificación de los ejercicios a mano
mano_comprueba_todos(["INTERP_LINEAL", "INTERP_CUADRATICA", "ERROR_INTERP"])

In [ ]:
#@title 📝 2 · Verificación de todas las preguntas (no envía nada)
puntos, maximo = calificar()

In [ ]:
#@title 📤 3 · Envío de las respuestas
enviar(email)

In [ ]:
#@title 🔎 4 · Verificación de lo que ya está enviado (no envía nada)
consultar_calificacion()